# 데이터 탐색
1. XML 파일 로드
2. XML 구조 탐색
3. 주요 태그 확인
4. annotation 구조 이해
5. JSON label 확인
6. extract_nodule_info() 구현
7. 테스트

In [3]:
# =========================================================
# 데이터 경로 확인
# =========================================================
import os

PATH = "/home/hdo/lidc_project/data"
BASE_PATH = "/home/hdo/lidc_project/data/lidc-idri"

print(os.listdir(PATH))
print(os.listdir(BASE_PATH))

['lidc-idri', 'processed']
['LIDC-XML-only', 'manifest-1600709154662', 'slices_png', 'lidc-idri-nodule-counts-6-23-2015.xlsx', 'slices', 'tcia-diagnosis-data-2012-04-20.xlsx', 'nodule_malignancy_scores.json']


In [4]:
# =========================================================
# XML 파일 수집 
# - tcia-lidc-xml 경로 지정
# =========================================================
import glob

xml_files = sorted(glob.glob(
    os.path.join(
        BASE_PATH,
        "LIDC-XML-only/tcia-lidc-xml/**/*.xml"
    ),
    recursive=True
))

print("Number of XML files:", len(xml_files))
print(xml_files[:5])

Number of XML files: 1318
['/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/158.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/159.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/160.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/161.xml', '/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/162.xml']


In [5]:
import xml.etree.ElementTree as ET
# XML 파일이 몇 개 발견됐는지 확인 (정상이면 약 300개)
print(len(xml_files))

# LIDC XML 파일 내부 태그는 네임스페이스가 붙어있음
# 예: <SeriesInstanceUid xmlns="http://www.nih.gov"> 형태
# 그냥 root.find('SeriesInstanceUid')하면 못 찾음 → ns 딕셔너리로 해결
ns = {'ns': 'http://www.nih.gov'}

for file in xml_files:

    # 파일 경로에서 마지막 7글자만 추출 (예: '001.xml')
    # ⚠️ 위험: 파일명이 7자 초과면 잘림, 경로 구분자 방식에도 취약
    # 예: '0100.xml' → '100.xml' 로 잘려서 잘못된 식별자가 됨
    file_name = file[-7:]

    # XML 파일을 파싱해서 트리 구조로 읽음
    tree = ET.parse(file)

    # 트리의 최상단(루트) 요소를 가져옴
    # LIDC XML은 <LidcReadMessage>가 루트
    root = tree.getroot()

    # .// → 루트 하위 어디서든 탐색
    # ns:SeriesInstanceUid → 네임스페이스 포함해서 태그 찾기
    series_uid = root.find('.//ns:SeriesInstanceUid', ns)

    # SeriesInstanceUid가 없는 XML은 건너뜀
    # (비정상 파일 방어)
    if series_uid is not None:
        # 파일명과 Series UID를 출력
        # 이걸 보고 눈으로 매칭 확인하려는 의도
        print(file_name, series_uid.text)

1318
158.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824
159.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.131939324905446238286154504249
160.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.213233719488315954975724605159
161.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327
162.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.311102747717834916358933873912
163.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.166112018536246672777968137997
164.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.151647338241909635299641922057
165.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.207201727479884428632451006739
166.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.400871509992541458184881866614
167.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.453946099750629491201946672998
168.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.115689755198961598722158240121
069.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636438705059720603192
071.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.619372068417051974713149104919
072.xml 1.3.6.1.4.1.14519.5.2.1.6279.6001.1707

3. 전체 태그 구조 파악

In [6]:
# =========================================================
# root 확인
# =========================================================
import xml.etree.ElementTree as ET

tree = ET.parse(xml_files[0])
root = tree.getroot()

print(root.tag)

{http://www.nih.gov}LidcReadMessage


In [7]:
# root 아래 태그
print("\n===== ROOT CHILDREN =====")

for child in root:
    print(child.tag)


===== ROOT CHILDREN =====
{http://www.nih.gov}ResponseHeader
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession
{http://www.nih.gov}readingSession


In [8]:
# child 아래 구조 확인
print("\n===== READING SESSION STRUCTURE =====")

for child in root:

    if "readingSession" in child.tag:

        for sub in child:
            print(sub.tag)

        break


===== READING SESSION STRUCTURE =====
{http://www.nih.gov}annotationVersion
{http://www.nih.gov}servicingRadiologistID
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}nonNodule
{http://www.nih.gov}nonNodule


In [9]:
# 결절 내부 구조 분석
print("\n===== FIRST NODULE DETAIL =====")

for elem in root.iter():

    if "unblindedReadNodule" in elem.tag:

        for child in elem:

            print("\nCHILD TAG:", child.tag)

            for sub in child.iter():

                print("   ", sub.tag, ":", sub.text)

        break


===== FIRST NODULE DETAIL =====

CHILD TAG: {http://www.nih.gov}noduleID
    {http://www.nih.gov}noduleID : 0

CHILD TAG: {http://www.nih.gov}roi
    {http://www.nih.gov}roi : 
    
    {http://www.nih.gov}imageZposition : 1604.5
    {http://www.nih.gov}imageSOP_UID : 1.3.6.1.4.1.14519.5.2.1.6279.6001.192028152416081898151427003898
    {http://www.nih.gov}inclusion : TRUE
    {http://www.nih.gov}edgeMap : None
    {http://www.nih.gov}xCoord : 177
    {http://www.nih.gov}yCoord : 353


실제 필요한 정보 가져오기

In [10]:
# =========================================================
# SeriesInstanceUid 찾기
# =========================================================

print("\n===== SERIES INSTANCE UID =====")

for elem in root.iter():

    if "SeriesInstanceUid" in elem.tag:

        print("Series UID:", elem.text)


# =========================================================
# malignancy 찾기
# =========================================================

print("\n===== MALIGNANCY =====")

for elem in root.iter():

    if "malignancy" in elem.tag:

        print("malignancy:", elem.text)


# =========================================================
# imageZposition 찾기
# =========================================================

print("\n===== IMAGE Z POSITION =====")

z_count = 0

for elem in root.iter():

    if "imageZposition" in elem.tag:

        print("z:", elem.text)

        z_count += 1

        if z_count > 10:
            break


===== SERIES INSTANCE UID =====
Series UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824

===== MALIGNANCY =====
malignancy: 3
malignancy: 3
malignancy: 4
malignancy: 5
malignancy: 4
malignancy: 4
malignancy: 5
malignancy: 1
malignancy: 5
malignancy: 5
malignancy: 3
malignancy: 3
malignancy: 3

===== IMAGE Z POSITION =====
z: 1604.5
z: 1550.5
z: 1547.5
z: 1544.5
z: 1541.5
z: 1538.5
z: 1535.5
z: 1487.5
z: 1484.5
z: 1481.5
z: 1478.5


In [11]:
# =========================================================
# edgeMap 좌표 확인
# =========================================================

print("\n===== EDGE MAP COORDINATES =====")

edge_count = 0

for elem in root.iter():

    if "edgeMap" in elem.tag:

        x = None
        y = None

        for child in elem:

            if "xCoord" in child.tag:
                x = child.text

            if "yCoord" in child.tag:
                y = child.text

        print(f"(x={x}, y={y})")

        edge_count += 1

        if edge_count > 20:
            break


===== EDGE MAP COORDINATES =====
(x=177, y=353)
(x=364, y=172)
(x=364, y=171)
(x=364, y=170)
(x=365, y=169)
(x=365, y=168)
(x=365, y=167)
(x=365, y=166)
(x=365, y=165)
(x=365, y=164)
(x=365, y=163)
(x=364, y=164)
(x=363, y=164)
(x=362, y=165)
(x=361, y=166)
(x=360, y=167)
(x=359, y=168)
(x=359, y=169)
(x=358, y=170)
(x=359, y=171)
(x=360, y=171)


## annotation 구조 확인

In [12]:
# =========================================================
# readingSession 개수 확인
# =========================================================

print("\n===== READING SESSION COUNT =====")

session_count = 0

for elem in root.iter():

    if "readingSession" in elem.tag:

        session_count += 1

print("Number of reading sessions:", session_count)


# =========================================================
# unblindedReadNodule 개수 확인
# =========================================================

print("\n===== NODULE COUNT =====")

nodule_count = 0

for elem in root.iter():

    if "unblindedReadNodule" in elem.tag:

        nodule_count += 1

print("Number of nodules:", nodule_count)



===== READING SESSION COUNT =====
Number of reading sessions: 4

===== NODULE COUNT =====
Number of nodules: 26


In [13]:
# 실제 UID 기준 데이터셋 개수 확인
uid_set = set()

for xml_path in xml_files:

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        for elem in root.iter():

            if "SeriesInstanceUid" in elem.tag:

                uid_set.add(elem.text)
                break

    except:
        continue

print("Unique CT Series:", len(uid_set))

Unique CT Series: 1018


하나씩 탐색했던 XML parsing 코드를
"실제 데이터 추출 함수" 형태로 정리
- Series UID 찾기
- malignancy 찾기
- edgeaMap 좌표 찾기
=> 하나의 함수로 통합

In [14]:
def extract_nodule_info(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    series_uid = None

    # -----------------------------------------------------
    # Series UID
    # -----------------------------------------------------

    for elem in root.iter():

        if "SeriesInstanceUid" in elem.tag:

            series_uid = elem.text
            break

    nodules = []

    # -----------------------------------------------------
    # Nodule Parsing
    # -----------------------------------------------------

    for nodule in root.iter():

        if "unblindedReadNodule" not in nodule.tag:
            continue

        malignancy = None
        coords = []

        # characteristic 내부 탐색
        for sub in nodule.iter():

            # malignancy
            if "malignancy" in sub.tag:
                malignancy = sub.text

            # edgeMap
            if "edgeMap" in sub.tag:

                x = None
                y = None

                for child in sub:

                    if "xCoord" in child.tag:
                        x = child.text

                    if "yCoord" in child.tag:
                        y = child.text

                coords.append((x, y))

        nodules.append({
            "series_uid": series_uid,
            "malignancy": malignancy,
            "coords": coords
        })

    return nodules

In [15]:
sample = extract_nodule_info(xml_files[0])

print("Number of nodules:", len(sample))

print(sample[0])

Number of nodules: 26
{'series_uid': '1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824', 'malignancy': None, 'coords': [('177', '353')]}


In [ ]:

def extract_nodule_info(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # =========================================================
    # Series UID
    # =========================================================
    series_uid = None

    for elem in root.iter():

        if "SeriesInstanceUid" in elem.tag:

            series_uid = elem.text
            break

    # =========================================================
    # Patient Data
    # =========================================================
    patient_data = {

        "series_uid": series_uid,

        "annotations": {}
    }

    nodule_idx = 0

    # =========================================================
    # Nodule Parsing
    # =========================================================
    for nodule in root.iter():

        if "unblindedReadNodule" not in nodule.tag:
            continue

        malignancies = []
        rois = []

        # -----------------------------------------------------
        # Malignancy
        # -----------------------------------------------------
        for sub in nodule.iter():

            if "malignancy" in sub.tag:

                try:
                    malignancies.append(int(sub.text))

                except:
                    pass

        # -----------------------------------------------------
        # ROI Parsing
        # -----------------------------------------------------
        for roi in nodule.iter():

            if "roi" not in roi.tag:
                continue

            z = None
            polygon = []

            for sub in roi:

                # z position
                if "imageZposition" in sub.tag:

                    try:
                        z = float(sub.text)

                    except:
                        z = None

                # edgeMap
                if "edgeMap" in sub.tag:

                    x = None
                    y = None

                    for child in sub:

                        if "xCoord" in child.tag:

                            try:
                                x = int(child.text)

                            except:
                                x = None

                        if "yCoord" in child.tag:

                            try:
                                y = int(child.text)

                            except:
                                y = None

                    # 좌표 저장
                    if x is not None and y is not None:

                        polygon.append((x, y))

            # ROI 저장
            rois.append({

                "z": z,

                "polygon": polygon
            })

        # -----------------------------------------------------
        # Nodule 저장
        # -----------------------------------------------------
        if len(malignancies) == 0:
            continue
        
        patient_data["annotations"][f"annotation_{nodule_idx}"] = {

            "malignancy": malignancies,

            "rois": rois
        }

        nodule_idx += 1

    return series_uid, patient_data


# =========================================================
# DATASET BUILD
# =========================================================
dataset = {}

for xml_path in xml_files:

    try:

        patient_id, patient_data = extract_nodule_info(xml_path)

        if patient_id is None:
            continue

        # 이미 존재하는 환자면 nodule 추가
        if patient_id in dataset:

            existing_annotations = dataset[patient_id]["annotations"]
            new_annotations = patient_data["annotations"]

            offset = len(existing_annotations)

            for _, annotation_data in new_annotations.items():

                existing_annotations[f"annotation_{offset}"] = annotation_data

                offset += 1

        else:

            dataset[patient_id] = patient_data

    except Exception as e:

        print("ERROR:", xml_path)
        print(e)


In [31]:
# =========================================================
# 확인용 출력 코드
# =========================================================
print(len(dataset))



# 첫 번째 환자 확인
first_patient = list(dataset.keys())[0]
print(first_patient)


1018
1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824


In [32]:
# 첫 번째 환자 전체 출력
pprint(dataset[first_patient])


{'annotations': {'annotation_0': {'malignancy': [3],
                                  'rois': [{'polygon': [(364, 172),
                                                        (364, 171),
                                                        (364, 170),
                                                        (365, 169),
                                                        (365, 168),
                                                        (365, 167),
                                                        (365, 166),
                                                        (365, 165),
                                                        (365, 164),
                                                        (365, 163),
                                                        (364, 164),
                                                        (363, 164),
                                                        (362, 165),
                                                        (361, 1

In [33]:
for patient_id, patient_data in dataset.items():

    mals = []

    for ann_data in patient_data["annotations"].values():

        mals.append(ann_data["malignancy"])

    if len(mals) > 1:

        print("\nPATIENT:", patient_id)
        print("MALIGNANCIES:", mals)

        break


PATIENT: 1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824
MALIGNANCIES: [[3], [3], [4], [5], [4], [4], [5], [1], [5], [5], [3], [3], [3]]


In [ ]:
from pprint import pprint

patient_id = list(dataset.keys())[0]

print("PATIENT ID:")
print(patient_id)

print("\n===== MALIGNANCY LIST =====")

for nodule_id, nodule_data in dataset[patient_id]["nodules"].items():

    print(
        nodule_id,
        "->",
        nodule_data["malignancy"]
    )

PATIENT ID:
1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824

===== MALIGNANCY LIST =====
nodule_0 -> 3
nodule_1 -> 3
nodule_2 -> 4
nodule_3 -> 5
nodule_4 -> 4
nodule_5 -> 4
nodule_6 -> 5
nodule_7 -> 1
nodule_8 -> 5
nodule_9 -> 5
nodule_10 -> 3
nodule_11 -> 3
nodule_12 -> 3


In [24]:
for nodule_id, nodule_data in dataset[patient_id]["nodules"].items():

    print("\n====================")

    print("NODULE:", nodule_id)

    pprint(nodule_data)

    print("====================")


NODULE: nodule_0
{'malignancy': 3,
 'rois': [{'polygon': [(364, 172),
                       (364, 171),
                       (364, 170),
                       (365, 169),
                       (365, 168),
                       (365, 167),
                       (365, 166),
                       (365, 165),
                       (365, 164),
                       (365, 163),
                       (364, 164),
                       (363, 164),
                       (362, 165),
                       (361, 166),
                       (360, 167),
                       (359, 168),
                       (359, 169),
                       (358, 170),
                       (359, 171),
                       (360, 171),
                       (361, 170),
                       (362, 169),
                       (363, 169),
                       (363, 171),
                       (364, 172)],
           'z': 1550.5},
          {'polygon': [(352, 183),
                       (353,